# Cálculo Diferencial e Integral — caderno de consulta e prática

Material de estudo orientado ao uso de Python para calcular, verificar e interpretar resultados de Cálculo Diferencial e Integral.

## Como usar
- Execute as células na ordem, especialmente a seção de configuração.
- Use `SymPy` para cálculos simbólicos e `NumPy`/`Matplotlib` para aproximações e gráficos.
- Um resultado calculado não dispensa interpretar domínio, hipóteses e significado.
- Compare resultados simbólicos com aproximações numéricas ou gráficos quando isso ajudar a validar a interpretação.

**Instalação, se necessário:** `pip install sympy numpy matplotlib scipy`

## Índice
1. Configuração e funções
2. **Tema 2 — Limites, continuidade e assíntotas: resolução de exercícios**
   - escolha do bloco por tipo de exercício; funções por partes; continuidade; assíntotas; conferência
3. **Tema 3 — Derivadas: conceitos, propriedades e cálculos**
4. **Tema 4 — Derivadas: aplicações**
5. **Tema 5 — Integrais: conceitos, propriedades e técnicas de integração**
6. **Tema 6 — Integrais: aplicações**
7. Checklist de escolha da ferramenta e exercícios rápidos

## 1. Configuração

`SymPy` trabalha com expressões simbólicas; isso é útil para obter formas exatas. `NumPy` e `Matplotlib` ajudam a explorar o comportamento numérico e visual das funções.

In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

x, h, a, C = sp.symbols('x h a C', real=True)
e = sp.exp(1)
pi = sp.pi
inf = sp.oo
sp.init_printing()

### Funções e domínio
Uma função pode ser definida simbolicamente e depois avaliada em valores específicos. Atenção: uma expressão algébrica pode não estar definida em todos os reais (por exemplo, denominadores nulos e logaritmos de números não positivos).

In [ ]:
f = (x**2 - 1)/(x - 1)
sp.simplify(f), sp.solve(sp.denom(f), x)

## 2. Limites, continuidade e assíntotas — resolução de exercícios

**Escolha o bloco pelo enunciado.** Execute a configuração inicial (seção 1) e depois apenas o bloco necessário. Cada bloco possui suas próprias entradas; altere as linhas identificadas como **ENTRADAS**.

| O exercício pede... | Execute |
|---|---|
| Limite de uma expressão em um número | **2.1** |
| Limites laterais, inclusive com funções por partes | **2.2** |
| Limite de função por partes em ±∞ | **2.3** |
| Verificar continuidade ou encontrar um parâmetro | **2.4** |
| Assíntotas verticais, horizontais e oblíquas | **2.5** |
| Assíntotas de função por partes | **2.6** |
| Limite por fatoração, racionalização ou oscilação | **2.7** |
| Conferência numérica próxima de um ponto | **2.8** |

**Sintaxe:** `x**2` é x²; `sp.sqrt(x)` é √x; `sp.oo` é +∞; `-sp.oo` é −∞. Em `sp.Piecewise`, as condições são avaliadas na ordem indicada; termine com `(expressao, True)` somente se houver um caso padrão real.

### 2.1 Limite de uma expressão em um ponto

**ENTRADAS:** `f`, `ponto`. Para um limite bilateral, o resultado deve ser o mesmo pela esquerda e pela direita. Um resultado `sp.oo` ou `-sp.oo` descreve divergência infinita, não um limite real finito.

In [ ]:
# ENTRADAS
f = (x**2 - 4)/(x - 2)
ponto = 2

# RESULTADOS
esquerda = sp.limit(f,x,ponto,dir='-')
direita = sp.limit(f,x,ponto,dir='+')
print('Limite pela esquerda:',esquerda)
print('Limite pela direita:',direita)
print('Limite bilateral:',esquerda if sp.simplify(esquerda-direita)==0 else 'não existe (laterais diferentes)')

### 2.2 Limites laterais de função definida por partes

**ENTRADAS:** `f` e `ponto`. Escreva cada par como `(expressao, condição)`. Exemplo: `x≤2` e `x>2`. O SymPy escolhe a expressão correspondente a cada lado; o valor exatamente no ponto não determina o limite.

In [ ]:
# ENTRADAS — troque expressões, condições e ponto
f = sp.Piecewise(
    (x**2, x <= 2),
    (3*x - 2, x > 2)
)
ponto = 2

# RESULTADOS — calculamos a partir dos ramos ativos em cada lado
def ramo_lateral(funcao, ponto, lado):
    if not isinstance(funcao, sp.Piecewise):
        return funcao
    delta = sp.Rational(1,10**6)
    amostra = sp.sympify(ponto) + (-delta if lado == '-' else delta)
    for expressao, condicao in funcao.args:
        if condicao is True or condicao == sp.true or condicao.subs(x,amostra) == sp.true:
            return expressao
    raise ValueError('Não existe ramo definido nesse lado.')

esquerda = sp.limit(ramo_lateral(f,ponto,'-'),x,ponto,dir='-')
direita = sp.limit(ramo_lateral(f,ponto,'+'),x,ponto,dir='+')
print('Função:',f)
print('Limite pela esquerda:',esquerda)
print('Limite pela direita:',direita)
print('Limite bilateral:',esquerda if esquerda==direita else 'não existe')
print('Valor no ponto:',f.subs(x,ponto))

**Atenção:** a seleção automática de ramo acima é adequada a condições simples com desigualdades e pontos numéricos, como `x<=2`, `x>2`. Para condições com parâmetros, intervalos muito estreitos ou regras mais complexas, indique explicitamente a expressão de cada lado no bloco abaixo.

In [ ]:
# ENTRADAS — alternativa universal para limites laterais
expressao_esquerda = x**2
expressao_direita = 3*x - 2
ponto = 2

L_esq = sp.limit(expressao_esquerda,x,ponto,dir='-')
L_dir = sp.limit(expressao_direita,x,ponto,dir='+')
print('Esquerda:',L_esq,'| Direita:',L_dir)
print('Bilateral:',L_esq if L_esq==L_dir else 'não existe')

### 2.3 Limites no infinito de funções por partes

**ENTRADAS:** `f`. Para `x→+∞`, use o ramo que vale para valores grandes e positivos; para `x→−∞`, o ramo válido para valores muito negativos. Limites diferentes nos dois sentidos são permitidos.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    ((2*x**2+1)/(x**2+3),x>2),
    ((x+1)/(x-1),True)
)

# RESULTADOS — escolha o ramo válido na direção correspondente
ramo_mais = ramo_lateral(f,10**6,'+')
ramo_menos = ramo_lateral(f,-10**6,'-')
print('Limite em +infinito:',sp.limit(ramo_mais,x,sp.oo))
print('Limite em -infinito:',sp.limit(ramo_menos,x,-sp.oo))

### 2.4 Continuidade em um ponto e parâmetro desconhecido

Para continuidade em `x=a`, precisam coincidir: limite pela esquerda, limite pela direita e valor da função. **ENTRADAS:** altere `esq`, `dir`, `valor_no_ponto` e `ponto`. Para encontrar `k`, use `sp.solve` nas igualdades; se os limites laterais não puderem coincidir, não há solução.

In [ ]:
# ENTRADAS — exemplo: f(x)=x² se x<2, k se x=2, 3x-2 se x>2
k = sp.symbols('k',real=True)
esq = x**2
dir_ = 3*x-2
valor_no_ponto = k
ponto = 2

L_esq = sp.limit(esq,x,ponto,dir='-')
L_dir = sp.limit(dir_,x,ponto,dir='+')
solucoes = sp.solve([L_esq-L_dir,L_esq-valor_no_ponto],[k],dict=True)
print('Limite esquerdo:',L_esq,'| direito:',L_dir)
print('Parâmetros para continuidade:',solucoes if solucoes else 'nenhuma solução')

### 2.5 Assíntotas de uma função (não definida por partes)

- **Vertical `x=a`:** ao menos um limite lateral é `+∞` ou `−∞`. Procure candidatos nos zeros reais do denominador e em outros pontos de fronteira do domínio (ex.: logaritmos).
- **Horizontal `y=L`:** limite finito em `+∞` ou `−∞`; pode haver retas diferentes.
- **Oblíqua `y=mx+b`:** `m=lim f(x)/x` e `b=lim(f(x)-mx)` na mesma direção, com `m` e `b` finitos e `m≠0`. Confirme `lim[f(x)-(mx+b)]=0`.

**ENTRADAS:** `f`, `candidatos_verticais`. Um zero do denominador pode ser apenas um furo, não uma assíntota.

In [ ]:
# ENTRADAS
f = (2*x**2+1)/(x-1)
candidatos_verticais = [1]

# RESULTADOS
for a0 in candidatos_verticais:
    le = sp.limit(f,x,a0,dir='-')
    ld = sp.limit(f,x,a0,dir='+')
    vertical = any(v in (sp.oo,-sp.oo) for v in (le,ld))
    print(f'x={a0}: esquerda={le}, direita={ld}, assíntota vertical={vertical}')

for direcao in (sp.oo,-sp.oo):
    L = sp.limit(f,x,direcao)
    print('Direção:',direcao,'| limite:',L)
    if L.is_finite is True:
        print('  Horizontal: y =',L)
    else:
        m = sp.limit(f/x,x,direcao)
        if m.is_finite is True and m != 0:
            b = sp.limit(f-m*x,x,direcao)
            if b.is_finite is True:
                print('  Oblíqua: y =',sp.expand(m*x+b))
            else:
                print('  Não há assíntota oblíqua identificada.')
        else:
            print('  Não há assíntota oblíqua identificada.')

### 2.6 Assíntotas de funções por partes

**ENTRADAS:** `f`, `pontos_de_transicao` e `outros_candidatos_verticais`. Investigue cada transição pelos dois lados: um dos ramos pode divergir enquanto o outro permanece finito. Para assíntotas horizontais e oblíquas, use **o ramo ativo na direção** `+∞` ou `−∞`. Inclua também polos interiores de cada ramo em `outros_candidatos_verticais` quando pertencerem ao domínio desse ramo em uma vizinhança unilateral.

In [ ]:
# ENTRADAS
f = sp.Piecewise(
    ((x**2+1)/(x-2),x>2),
    ((3*x+1)/(x+1),True)
)
pontos_de_transicao = [2]
outros_candidatos_verticais = [-1]

# RESULTADOS
for a0 in sorted(set(pontos_de_transicao+outros_candidatos_verticais)):
    expr_esq = ramo_lateral(f,a0,'-')
    expr_dir = ramo_lateral(f,a0,'+')
    le = sp.limit(expr_esq,x,a0,dir='-')
    ld = sp.limit(expr_dir,x,a0,dir='+')
    print(f'x={a0}: esquerda={le}, direita={ld}, vertical={any(v in (sp.oo,-sp.oo) for v in (le,ld))}')

for direcao,ramo in ((sp.oo,ramo_lateral(f,10**6,'+')),
                     (-sp.oo,ramo_lateral(f,-10**6,'-'))):
    L = sp.limit(ramo,x,direcao)
    print('Direção',direcao,'| limite:',L)
    if L.is_finite is True:
        print('  Horizontal: y =',L)
    else:
        m = sp.limit(ramo/x,x,direcao)
        if m.is_finite is True and m != 0:
            b = sp.limit(ramo-m*x,x,direcao)
            if b.is_finite is True:
                print('  Oblíqua: y =',sp.expand(m*x+b))
            else:
                print('  Nenhuma oblíqua identificada.')
        else:
            print('  Nenhuma oblíqua identificada.')

### 2.7 Indeterminações, fatoração, racionalização e oscilação

**ENTRADAS:** substitua `f` e `ponto`. `sp.factor` ajuda a identificar fatores comuns; `sp.radsimp` e `sp.simplify` podem ajudar em expressões com radicais. A forma `0/0` não é o valor do limite. Expressões oscilatórias, como `sin(1/x)`, podem não ter limite, enquanto `x² sin(1/x)` tende a zero pelo confronto.

In [ ]:
# ENTRADAS
f = (sp.sqrt(x+1)-1)/x
ponto = 0

# RESULTADOS
print('Expressão original:',f)
print('Forma simplificada:',sp.simplify(f))
print('Fatoração (quando aplicável):',sp.factor(f))
print('Limite esquerdo:',sp.limit(f,x,ponto,dir='-'))
print('Limite direito:',sp.limit(f,x,ponto,dir='+'))
print('Exemplo oscilatório limitado:',sp.limit(x**2*sp.sin(1/x),x,0))

### 2.8 Conferência numérica perto de um ponto

**ENTRADAS:** `f`, `ponto`. A tabela mostra aproximações laterais; **não prova** a existência do limite. Evite avaliar exatamente no ponto quando houver divisão por zero. Para funções por partes, `sp.lambdify` pode avaliar os ramos, mas expressões com singularidades exigem cuidado.

In [ ]:
# ENTRADAS
f = (x**2-4)/(x-2)
ponto = 2
distancias = [1e-1,1e-2,1e-3,1e-4]

# RESULTADOS
for d in distancias:
    esquerda = sp.N(f.subs(x,sp.Float(ponto-d)),10)
    direita = sp.N(f.subs(x,sp.Float(ponto+d)),10)
    print(f'distância={d:g}: esquerda={esquerda}, direita={direita}')

## 3. Derivadas: conceitos, propriedades e cálculos

A derivada representa a **taxa de variação instantânea** e, quando existe, o coeficiente angular da reta tangente. Pela definição, `f'(a) = lim(h→0) [f(a+h)-f(a)]/h`. Uma função derivável em um ponto é contínua nele; o contrário não é garantido (ex.: `abs(x)` em zero).

**Roteiro de consulta:** identifique a expressão e seu domínio; escolha a regra; use `sp.diff`; simplifique; substitua o ponto apenas depois de derivar. Verifique separadamente pontos de descontinuidade e de não diferenciabilidade.

In [ ]:
# Definição por limite versus derivação direta
h = sp.symbols('h', real=True)
f = x**3 - 5*x**2 + 2*x + 7
a = sp.Integer(2)
pela_definicao = sp.limit((f.subs(x, a+h)-f.subs(x, a))/h, h, 0)
pela_regra = sp.diff(f, x).subs(x, a)
sp.Eq(pela_definicao, pela_regra), sp.diff(f, x)

### 3.1 Regras e funções elementares

- Linearidade: `(c f + g)' = c f' + g'`; potência: `(x**n)' = n*x**(n-1)` nos pontos em que a expressão é diferenciável.
- Produto: `(fg)' = f'g + fg'`; quociente: `(f/g)' = (f'g - fg')/g**2` onde `g ≠ 0`.
- Cadeia: `(F(g(x)))' = F'(g(x))*g'(x)`; a derivada da **função interna** não pode ser omitida.
- `sin → cos`, `cos → -sin`, `tan → sec²`, `exp → exp`, `log(x) → 1/x` (para logaritmo real, `x>0`); `(a**x)' = a**x*log(a)` para `a>0`.
- Inversas: `(f⁻¹)'(y) = 1/f'(f⁻¹(y))` quando a inversa local existe e `f' ≠ 0`.

In [ ]:
exemplos = {
    'produto': (x**2+1)*sp.sin(x),
    'quociente': (x**2+1)/(x-1),
    'cadeia': sp.sin(x**2+3*x),
    'exponencial': sp.exp(2*x),
    'logaritmo': sp.log(x**2+1),
    'trigonometrica': sp.tan(x),
    'potencia': x**sp.Rational(3,2)
}
{nome: sp.simplify(sp.diff(expr, x)) for nome,expr in exemplos.items()}

### 3.2 Continuidade, derivadas laterais e pontos angulosos

Em funções por partes, calcule os limites laterais e o valor no ponto. Mesmo quando os três coincidem, as derivadas laterais podem diferir. Em `|x|`, por exemplo, as inclinações são `-1` à esquerda e `1` à direita de zero.

In [ ]:
g = sp.Abs(x)
der_esq = sp.limit((g.subs(x, 0+h)-g.subs(x, 0))/h, h, 0, dir='-')
der_dir = sp.limit((g.subs(x, 0+h)-g.subs(x, 0))/h, h, 0, dir='+')
{'continua_em_zero': sp.limit(g, x, 0)==g.subs(x,0),
 'derivada_esquerda': der_esq, 'derivada_direita': der_dir,
 'derivavel_em_zero': der_esq==der_dir}

### 3.3 Derivação implícita, inversa e ordens superiores

Uma relação `F(x,y)=0` pode definir `y` implicitamente. `sp.idiff` calcula `dy/dx` tratando `y` como função de `x`. A segunda derivada é `sp.diff(f,x,2)`; derivadas de ordem superior seguem a mesma sintaxe. Para a função inversa, a identidade `f(f⁻¹(y))=y` fundamenta a regra recíproca.

In [ ]:
y = sp.symbols('y', real=True)
F = x**2 + y**2 - 25
dy_dx = sp.idiff(F, y, x)
f = x**4-2*x**2
{'derivada_implicita': dy_dx,
 'inclinacao_em_(3,4)': dy_dx.subs({x:3,y:4}),
 'primeira': sp.diff(f,x), 'segunda': sp.diff(f,x,2),
 'terceira': sp.diff(f,x,3),
 'derivada_da_inversa_de_exp': sp.diff(sp.log(y),y)}

## 4. Derivadas: aplicações

**Como identificar o procedimento:** inclinação ou tangente → `f'(a)`; normal → inclinação `-1/f'(a)` quando `f'(a)≠0`; velocidade → derivada da posição; aceleração → segunda derivada; extremos → pontos críticos e comparação; concavidade → sinal de `f''`; otimização → modelar objetivo, domínio e restrições.

A derivada fornece candidatos, **não** garante extremos. Pontos críticos interiores ocorrem onde `f'=0` ou onde `f'` não existe (com `f` definida).

### 4.1 Retas tangente e normal

No ponto `x=a`, a tangente é `y=f(a)+f'(a)(x-a)`. Se a tangente for horizontal, a normal é vertical (`x=a`); se a derivada não existir, é necessário analisar o caso geométrico, sem aplicar automaticamente a fórmula.

In [ ]:
f = x**2+1
a = sp.Integer(2)
m = sp.diff(f,x).subs(x,a)
tangente = sp.expand(f.subs(x,a)+m*(x-a))
normal = sp.expand(f.subs(x,a)-sp.Rational(1,1)/m*(x-a)) if m != 0 else sp.Eq(x,a)
{'ponto':(a,f.subs(x,a)), 'inclinacao':m, 'tangente_y':tangente, 'normal_y':normal}

### 4.2 Taxas de variação e taxas relacionadas

Se `s(t)` é posição, `v(t)=s'(t)` e `a(t)=s''(t)`. Em taxas relacionadas, escreva uma equação entre grandezas, derive em relação ao **tempo** e só então substitua os valores conhecidos. As unidades devem ser consistentes.

In [ ]:
t = sp.symbols('t', real=True)
s = 2*t**3-9*t**2+12*t
v, aceleracao = sp.diff(s,t), sp.diff(s,t,2)
# Área de círculo A=pi*r(t)^2; quando r=3 e dr/dt=2:
r_t = sp.Function('r')(t)
A = sp.pi*r_t**2
dA_dt = sp.diff(A,t)
{'velocidade':v,'aceleracao':aceleracao,
 'dA_dt':dA_dt,
 'taxa_area_no_instante':dA_dt.subs({r_t:3,sp.diff(r_t,t):2})}

### 4.3 Crescimento, extremos e testes de derivadas

O sinal de `f'` indica crescimento (`+`) ou decrescimento (`-`) em intervalos do domínio. O **teste da primeira derivada** classifica um ponto crítico quando há troca de sinal: `+→-` máximo local, `-→+` mínimo local. Pelo **teste da segunda derivada**, se `f'(c)=0` e `f''(c)>0`, há mínimo local; se `f''(c)<0`, máximo local; se `f''(c)=0`, o teste é inconclusivo.

**Teorema de Fermat:** em um extremo local interior onde a derivada existe, `f'(c)=0`. **Teorema de Weierstrass:** função contínua em intervalo fechado e limitado atinge máximo e mínimo globais. Para extremos globais, compare valores em pontos críticos, pontos não diferenciáveis e extremos do intervalo.

In [ ]:
f = x**3-6*x**2+9*x+1
d1, d2 = sp.diff(f,x), sp.diff(f,x,2)
criticos = sp.solve(sp.Eq(d1,0),x)
analise = [{'x':c,'f(x)':f.subs(x,c),'f_segunda':d2.subs(x,c),
            'tipo':('minimo local' if d2.subs(x,c)>0 else 'maximo local' if d2.subs(x,c)<0 else 'inconclusivo')}
           for c in criticos]
{'primeira_derivada':d1,'segunda_derivada':d2,'pontos_criticos':analise,
 'intervalos_crescentes':sp.solve_univariate_inequality(d1>0,x,relational=False),
 'intervalos_decrescentes':sp.solve_univariate_inequality(d1<0,x,relational=False)}

### 4.4 Concavidade e inflexão

`f''>0` indica concavidade para cima; `f''<0`, para baixo. Um ponto de inflexão exige **mudança efetiva de concavidade**; apenas resolver `f''=0` produz candidatos.

In [ ]:
f = x**3-6*x**2+9*x+1
d2 = sp.diff(f,x,2)
candidatos_inflexao = sp.solve(sp.Eq(d2,0),x)
{'segunda_derivada':d2,'candidatos':candidatos_inflexao,
 'concavidade_para_cima':sp.solve_univariate_inequality(d2>0,x,relational=False),
 'concavidade_para_baixo':sp.solve_univariate_inequality(d2<0,x,relational=False)}

### 4.5 Otimização em intervalo fechado

Procedimento: (1) formule a função objetivo; (2) determine o domínio admissível; (3) calcule a derivada; (4) encontre candidatos interiores, incluindo pontos não diferenciáveis; (5) compare com os extremos do intervalo. Exemplo: maximizar a área de um retângulo de perímetro 20, com lados `x` e `10-x`, para `0≤x≤10`.

In [ ]:
area = x*(10-x)
candidatos = [sp.Integer(0),sp.Integer(10)] + [
 c for c in sp.solve(sp.Eq(sp.diff(area,x),0),x) if c.is_real and 0<=c<=10]
valores = {c:area.subs(x,c) for c in candidatos}
{'funcao_objetivo':area,'candidatos_e_areas':valores,
 'maximo':max(valores.items(),key=lambda item:item[1]),
 'minimo':min(valores.items(),key=lambda item:item[1])}

### 4.6 Visualização da análise de uma função

Altere `f` e o intervalo de visualização para estudar outros exemplos. O gráfico ajuda a interpretar, mas não substitui a verificação algébrica de domínio, sinais e extremos.

In [ ]:
f = x**3-6*x**2+9*x+1
xs = np.linspace(-0.5,4.5,500)
fn = sp.lambdify(x,f,'numpy')
criticos = [float(c) for c in sp.solve(sp.Eq(sp.diff(f,x),0),x) if c.is_real]
inflexoes = [float(c) for c in sp.solve(sp.Eq(sp.diff(f,x,2),0),x) if c.is_real]
plt.figure(figsize=(9,5))
plt.plot(xs,fn(xs),label='f(x)')
for c in criticos:
    plt.scatter([c],[float(f.subs(x,c))],marker='o',s=65,label=f'crítico x={c:g}')
for c in inflexoes:
    plt.scatter([c],[float(f.subs(x,c))],marker='x',s=90,label=f'inflexão x={c:g}')
plt.axhline(0,color='gray',linewidth=.7)
plt.grid(True);plt.legend();plt.xlabel('x');plt.ylabel('f(x)');plt.show()

## 5. Integrais: conceitos, propriedades e técnicas

A **integral indefinida** fornece uma família de primitivas: se `F'=f`, então `∫f(x)dx=F(x)+C`. Uma **condição inicial** determina `C`. A **integral definida** produz um valor numérico (com sinal), interpretado como limite de somas de Riemann quando a função é integrável.

**Escolha do procedimento:** integral imediata → `sp.integrate`; função composta com derivada interna → substituição; produto de funções → partes; função racional → divisão polinomial e frações parciais. Verifique a derivada da primitiva e respeite o domínio.

In [ ]:
# Primitiva, constante de integração e condição inicial F(1)=5
f = 3*x**2+2*x
F0 = sp.integrate(f,x)
C = sp.symbols('C')
constante = sp.solve(sp.Eq((F0+C).subs(x,1),5),C)[0]
F = F0+constante
{'familia':F0+C,'constante':constante,'primitiva_particular':F,
 'verificacao':sp.simplify(sp.diff(F,x)-f)}

### 5.1 Integrais imediatas e propriedades

Linearidade: `∫(a f+b g)dx = a∫f dx+b∫g dx`. Primitivas úteis: `∫x**n dx=x**(n+1)/(n+1)+C` para `n≠-1`; `∫1/x dx=ln|x|+C` em intervalos sem zero; `∫exp(x)dx=exp(x)+C`; `∫sin(x)dx=-cos(x)+C`; `∫cos(x)dx=sin(x)+C`.

In [ ]:
integrandos = [x**4, 1/x, sp.exp(x), sp.sin(x), sp.cos(x), 2*x+sp.cos(x)]
[(g,sp.integrate(g,x),sp.simplify(sp.diff(sp.integrate(g,x),x)-g)) for g in integrandos]

### 5.2 Soma de Riemann e integral definida

Particione `[a,b]` em `n` intervalos de largura `Δx=(b-a)/n`. A soma à direita é `Σ f(a+iΔx)Δx`; quando converge com a malha tendendo a zero, obtém-se a integral. **A integral pode ser negativa** em regiões abaixo do eixo; área geométrica não é a mesma coisa.

In [ ]:
f = x**2
a,b = sp.Integer(0),sp.Integer(2)
n,i = sp.symbols('n i',integer=True,positive=True)
dx = (b-a)/n
soma = sp.summation(f.subs(x,a+i*dx)*dx,(i,1,n))
{'soma_direita':sp.factor(soma),
 'limite':sp.limit(soma,n,sp.oo),
 'integral_exata':sp.integrate(f,(x,a,b))}

### 5.3 Teorema Fundamental do Cálculo (TFC)

Se `f` é contínua, a função `G(x)=∫_a^x f(t)dt` satisfaz `G'(x)=f(x)` (primeira parte). Se `F'=f`, então `∫_a^b f(x)dx=F(b)-F(a)` (segunda parte). Para limites variáveis, combine o TFC com a regra da cadeia.

In [ ]:
t = sp.symbols('t',real=True)
f_t = t**2+sp.sin(t)
G = sp.integrate(f_t,(t,0,x))
F = sp.integrate(f_t,t)
{'funcao_acumulada':G,'derivada_da_acumulada':sp.simplify(sp.diff(G,x)),
 'integral_0_a_2':sp.simplify(F.subs(t,2)-F.subs(t,0)),
 'conferencia':sp.integrate(f_t,(t,0,2)),
 'limite_variavel_x2':sp.simplify(sp.diff(sp.integrate(f_t,(t,0,x**2)),x))}

### 5.4 Substituição de variável

Procure uma expressão interna `u=g(x)` cuja derivada apareça no integrando. Transforme também `dx`; em integrais definidas, altere os limites para `u(a)` e `u(b)`, ou retorne à variável original antes de avaliar.

In [ ]:
u = sp.symbols('u',real=True)
integrando = 2*x*sp.cos(x**2)
via_u = sp.integrate(sp.cos(u),u).subs(u,x**2)
{'substituicao_u':'x**2','primitiva':via_u,
 'verificacao':sp.simplify(sp.diff(via_u,x)-integrando),
 'definida':sp.integrate(integrando,(x,0,sp.sqrt(sp.pi))),
 'limites_transformados':sp.integrate(sp.cos(u),(u,0,sp.pi))}

### 5.5 Integração por partes

Use `∫u dv = uv - ∫v du`, derivada da regra do produto. É especialmente útil para produtos envolvendo polinômios com exponenciais, logaritmos ou trigonométricas. A escolha de `u` deve simplificar a integral restante.

In [ ]:
u = x
dv = sp.exp(x)
du = sp.diff(u,x)
v = sp.integrate(dv,x)
resultado_partes = sp.simplify(u*v-sp.integrate(v*du,x))
{'integrando':u*dv,'resultado':resultado_partes,
 'verificacao':sp.simplify(sp.diff(resultado_partes,x)-u*dv)}

### 5.6 Frações parciais

Para `P(x)/Q(x)`, verifique se o grau do numerador é menor que o do denominador; caso contrário, divida os polinômios antes. Fatore `Q` e decomponha em frações simples. O SymPy oferece `sp.apart`; a decomposição e a integração dependem do domínio e dos fatores reais/complexos.

In [ ]:
racional = (2*x+3)/(x**2+3*x+2)
decomposta = sp.apart(racional,x)
primitiva = sp.integrate(decomposta,x)
impropria = (x**3+1)/(x**2-1)
{'fracao':racional,'decomposicao':decomposta,
 'primitiva':primitiva,
 'verificacao':sp.simplify(sp.diff(primitiva,x)-racional),
 'divisao_antes_de_decompor':sp.div(x**3+1,x**2-1,x)}

### 5.7 Verificação e limitações

`sp.integrate` pode devolver uma expressão especial ou uma integral não avaliada. Quando necessário, aproxime numericamente com `sp.N` ou técnicas numéricas e verifique o domínio. A constante `C` é omitida pela API simbólica; acrescente-a na interpretação de integrais indefinidas.

In [ ]:
integrando = x**4-3*x+sp.cos(x)
F = sp.integrate(integrando,x)
{'primitiva':F,'residuo_apos_derivar':sp.simplify(sp.diff(F,x)-integrando),
 'integral_definida_exata':sp.integrate(sp.sqrt(1+x**4),(x,0,1)),
 'valor_aproximado':sp.N(sp.integrate(sp.sqrt(1+x**4),(x,0,1)),12)}

## 6. Integrais: aplicações

**Mapa rápido:** área com sinal → `∫f`; área geométrica → `∫|f|`; área entre curvas → `∫|f-g|`; comprimento de arco → `∫sqrt(1+(f')²)`; área de superfície de revolução → `2π∫(raio)ds`; volume por seções → `∫A(x)dx`; sólidos de revolução → discos, arruelas ou cascas cilíndricas.

Antes de integrar, **desenhe/identifique a região, o intervalo, o eixo de rotação e as unidades**. Divida intervalos quando curvas se cruzam ou mudam de sinal.

### 6.1 Área líquida, área geométrica e área entre curvas

A integral definida soma áreas **com sinal**. Para área física entre gráfico e eixo, integre `|f|` (ou separe nas raízes). Entre curvas, integre `|f-g|`, particionando nas interseções; quando `f≥g` em todo intervalo, basta `∫(f-g)`.

In [ ]:
f = x-1
a,b = 0,3
liquida = sp.integrate(f,(x,a,b))
geometrica = sp.integrate(sp.Abs(f),(x,a,b))
superior,inferior = x,x**2
entre = sp.integrate(superior-inferior,(x,0,1))
{'area_com_sinal':liquida,'area_geometrica':geometrica,
 'intersecoes':sp.solve(sp.Eq(superior,inferior),x),
 'area_entre_curvas_0_a_1':entre}

### 6.2 Comprimento de arco

Para curva `y=f(x)` continuamente derivável em `[a,b]`, o comprimento é `L=∫_a^b sqrt(1+[f'(x)]²)dx`. A função comprimento acumulado desde `a` é `s(x)=∫_a^x sqrt(1+[f'(t)]²)dt`. Em curvas paramétricas, use `∫sqrt((dx/dt)²+(dy/dt)²)dt`.

In [ ]:
f = x**sp.Rational(3,2)
elemento_arco = sp.sqrt(1+sp.diff(f,x)**2)
comprimento = sp.integrate(elemento_arco,(x,0,4))
t = sp.symbols('t',real=True)
comprimento_parametrico = sp.integrate(sp.sqrt(sp.diff(t,t)**2+sp.diff(t**2,t)**2),(t,0,1))
{'ds_dx':elemento_arco,'comprimento_0_a_4':sp.simplify(comprimento),
 'comprimento_parametrico':comprimento_parametrico}

### 6.3 Área de superfície de revolução

Ao girar `y=f(x)≥0` em torno do eixo **x**, a área lateral é `S=2π∫_a^b f(x)sqrt(1+[f'(x)]²)dx`. Em torno do eixo **y**, para curva parametrizada por `x`, o raio é `|x|`: `S=2π∫|x|sqrt(1+[f'(x)]²)dx`. A fórmula calcula a superfície gerada pela curva, não tampas de um sólido.

In [ ]:
f = x  # segmento de reta y=x, de x=0 a x=2
a,b = 0,2
ds = sp.sqrt(1+sp.diff(f,x)**2)
S_eixo_x = sp.simplify(2*sp.pi*sp.integrate(f*ds,(x,a,b)))
S_eixo_y = sp.simplify(2*sp.pi*sp.integrate(x*ds,(x,a,b)))
{'superficie_ao_redor_de_x':S_eixo_x,'superficie_ao_redor_de_y':S_eixo_y}

### 6.4 Volumes por seções, discos e arruelas

Se a seção transversal perpendicular ao eixo `x` tem área `A(x)`, então `V=∫_a^b A(x)dx`. Para revolução em torno do eixo `x`: **discos** `V=π∫R(x)²dx`; **arruelas** `V=π∫[R(x)²-r(x)²]dx`. Exija `R≥r≥0` no intervalo; os raios são **distâncias ao eixo**, não valores arbitrários das funções.

In [ ]:
# Região entre y=x e o eixo x, 0<=x<=2, girada em torno do eixo x
V_disco = sp.pi*sp.integrate(x**2,(x,0,2))
# Região entre y=2 e y=x, 0<=x<=2, girada em torno do eixo x
V_arruela = sp.pi*sp.integrate(2**2-x**2,(x,0,2))
# Volume por seções quadradas de lado x
V_secoes = sp.integrate(x**2,(x,0,2))
{'discos':V_disco,'arruelas':V_arruela,'secoes_quadradas':V_secoes}

### 6.5 Cascas cilíndricas e rotação em torno de outro eixo

Para girar em torno do eixo **y** uma região sob `f(x)≥0`, com `0≤a≤b`, as cascas têm raio `x` e altura `f(x)`: `V=2π∫_a^b x f(x)dx`. Para um eixo vertical `x=c`, o raio é `|x-c|`; certifique-se de que a região não gere sobreposição na montagem das cascas. Escolha discos/arruelas ou cascas conforme a orientação das fatias.

In [ ]:
# Região sob y=x, entre x=0 e x=2, girada em torno do eixo y
V_cascas = 2*sp.pi*sp.integrate(x*x,(x,0,2))
# Conferência por discos perpendiculares ao eixo y: raio externo 2, interno y
y = sp.symbols('y',real=True)
V_arruelas_em_y = sp.pi*sp.integrate(4-y**2,(y,0,2))
{'cascas':V_cascas,'arruelas_no_eixo_y':V_arruelas_em_y,
 'coincidem':sp.simplify(V_cascas-V_arruelas_em_y)==0}

### 6.6 Aproximação numérica e checagens

Quando a primitiva não é elementar, a integral definida pode ser avaliada numericamente. Compare resultados simbólicos e numéricos, confira unidades (`comprimento`, `área`, `volume`) e confirme que áreas e volumes geométricos não sejam negativos. Uma regra do trapézio simples é útil para funções amostradas, mas a precisão depende da suavidade e da discretização.

In [ ]:
def regra_trapezios(func,a,b,n=1000):
    xs = np.linspace(a,b,n+1)
    ys = np.asarray(func(xs),dtype=float)
    return np.trapezoid(ys,xs) if hasattr(np,'trapezoid') else np.trapz(ys,xs)
f = x**2
exato = sp.integrate(f,(x,0,2))
aproximado = regra_trapezios(lambda z:z**2,0,2,1000)
{'exato':exato,'aproximado':aproximado,'erro_absoluto':abs(aproximado-float(exato))}

## 7. Escolha rápida da ferramenta

| Objetivo | Código típico |
|---|---|
| Limite simbólico | `sp.limit(expr, x, a)` |
| Limite lateral | `sp.limit(expr, x, a, dir='+' ou '-')` |
| Limite no infinito | `sp.limit(expr, x, sp.oo)` ou `-sp.oo` |
| Simplificar/fatorar | `sp.simplify(expr)`, `sp.factor(expr)` |
| Derivar | `sp.diff(expr, x)` |
| Derivada de ordem 2 | `sp.diff(expr, x, 2)` |
| Integrar sem limites | `sp.integrate(expr, x)` |
| Integral definida | `sp.integrate(expr, (x, a, b))` |
| Avaliar expressão | `expr.subs(x, valor)` |
| Gerar função numérica | `sp.lambdify(x, expr, 'numpy')` |

## Erros frequentes
- Tratar `0/0` como se fosse o valor do limite.
- Supor que o limite existe sem comparar os laterais.
- Confundir `f(a)` com `lim f(x)`.
- Concluir que toda descontinuidade gera assíntota vertical.
- Esquecer o domínio (por exemplo, `ln(x)` exige `x>0` nos reais).
- Em otimização, considerar apenas `f'=0` e ignorar extremos do intervalo ou pontos não diferenciáveis.
- Interpretar integral definida como área geométrica sem considerar o sinal.
- Esquecer `+C` em integrais indefinidas.

## Exercícios rápidos para testar o fluxo

Execute as células e confira se os resultados fazem sentido. Tente prever o resultado antes de executar.

1. Calcule `lim (x**2 - 4)/(x - 2)` quando `x→2`.
2. Verifique se `f(x)=(x**2-1)/(x-1)` tem limite em `x=1` e se está definida nesse ponto.
3. Encontre a derivada de `x**2*sin(x)`.
4. Calcule `∫_0^3 2x dx`.
5. Calcule a área entre `y=x` e `y=x**2` em `[0,1]`.

In [ ]:
# 1 e 2 — limites
L1 = sp.limit((x**2 - 4)/(x - 2), x, 2)
L2 = sp.limit((x**2 - 1)/(x - 1), x, 1)
valor_no_ponto = ((x**2 - 1)/(x - 1)).subs(x, 1)
L1, L2, valor_no_ponto

In [ ]:
# 3 a 5 — derivada e integrais
D3 = sp.diff(x**2 * sp.sin(x), x)
I4 = sp.integrate(2*x, (x, 0, 3))
I5 = sp.integrate(x - x**2, (x, 0, 1))
D3, I4, I5

## Síntese

Limites descrevem o comportamento de uma função perto de um ponto ou no infinito. Derivadas modelam taxas de variação instantânea e ajudam a analisar tangentes, crescimento, extremos e otimização. Integrais representam acumulação e permitem calcular áreas, deslocamentos, valores médios e volumes.

Ao resolver um problema computacionalmente, identifique a expressão e suas variáveis, verifique domínio e hipóteses, escolha a operação simbólica ou numérica adequada e interprete o resultado no contexto.